# 01 — Audio Extraction

**Purpose:** Pull the raw audio track out of the source video at its **native sample rate** — no resampling here.

## Why preserve native sample rate?
Most cameras, broadcast masters, and streaming encodes use **48 000 Hz** (not 44 100 Hz). Downsampling at this stage — before stem separation — would degrade the music bed *twice* (once here, once in assembly). We avoid that by:
1. Probing the actual stream SR with `ffprobe`
2. Extracting at native SR
3. Only resampling when a downstream model *requires* it (ASR needs 16 kHz; that resample happens in memory in notebook 04, never on disk)

The ElevenLabs TTS outputs 44 100 Hz PCM (Pro plan limit). At assembly time (notebook 07) we *upsample* the TTS audio to match the source SR — upsampling is non-destructive.

**Input:** `input/dub_original.mp4`  
**Output:** `intermediate/audio_extracted/source_audio.wav` at native SR  
**Metadata:** `intermediate/audio_extracted/meta.json`


In [ ]:
import importlib
missing = [p for p in ["librosa","soundfile"] if not importlib.util.find_spec(p)]
if missing:
    import subprocess, sys
    subprocess.run([sys.executable,"-m","pip","install","-q","ffmpeg-python","soundfile","librosa","tqdm"])
print("Ready.")

In [ ]:
import sys, os, json, subprocess, time
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
import librosa
import soundfile as sf
from IPython.display import Audio, display

sys.path.insert(0, os.path.abspath('..'))
from config import (
    SOURCE_VIDEO_PATH, AUDIO_EXTRACTED_DIR,
    LANGUAGE_DISPLAY, SOURCE_LANGUAGE,
)

OUTPUT_WAV  = os.path.join(AUDIO_EXTRACTED_DIR, 'source_audio.wav')
META_PATH   = os.path.join(AUDIO_EXTRACTED_DIR, 'meta.json')

print(f'Source video : {SOURCE_VIDEO_PATH}')
print(f'Exists       : {os.path.exists(SOURCE_VIDEO_PATH)}')
print(f'Output WAV   : {OUTPUT_WAV}')


In [ ]:
# ── Probe video metadata ──────────────────────────────────────────────────────
# We use ffprobe (bundled with ffmpeg) to read stream-level metadata without
# touching any audio samples. This gives us the authoritative source SR.
result = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-print_format', 'json',
     '-show_format', '-show_streams', SOURCE_VIDEO_PATH],
    capture_output=True, text=True, check=True
)
info = json.loads(result.stdout)
fmt  = info['format']

DURATION = float(fmt['duration'])
print(f'Duration : {DURATION:.2f}s  ({DURATION/60:.1f} min)')
print(f'Format   : {fmt["format_name"]}')
print(f'Size     : {int(fmt["size"])/1e6:.1f} MB')

SOURCE_SAMPLE_RATE = None
for s in info['streams']:
    codec  = s.get('codec_type', '')
    cname  = s.get('codec_name', '?')
    width  = s.get('width', '')
    height = s.get('height', '')
    sr     = s.get('sample_rate', '')
    print(f'  Stream #{s["index"]} {codec:6s}: {cname}  {width}{"x" if width else ""}{height}  {sr+" Hz" if sr else ""}')
    if codec == 'audio' and sr:
        SOURCE_SAMPLE_RATE = int(sr)

print(f'\nDetected source sample rate: {SOURCE_SAMPLE_RATE} Hz')
if SOURCE_SAMPLE_RATE not in (44100, 48000, 22050, 96000):
    print(f'WARNING: unusual sample rate {SOURCE_SAMPLE_RATE} — proceed with caution')


In [ ]:
# ── Extract audio at NATIVE sample rate ───────────────────────────────────────
# No -ar flag → ffmpeg preserves the source stream's SR exactly.
# PCM 16-bit signed little-endian (.wav) is lossless.
if os.path.exists(OUTPUT_WAV):
    print(f'Already extracted — skipping: {OUTPUT_WAV}')
    # Still need to confirm the SR of the cached file
    _, cached_sr = sf.read(OUTPUT_WAV, frames=1)
    print(f'Cached SR: {cached_sr} Hz')
    if SOURCE_SAMPLE_RATE is None:
        SOURCE_SAMPLE_RATE = cached_sr
else:
    t0 = time.time()
    cmd = [
        'ffmpeg', '-y',
        '-i', SOURCE_VIDEO_PATH,
        '-vn',                    # discard video stream
        '-acodec', 'pcm_s16le',   # 16-bit signed PCM (lossless)
        '-ac', '2',               # stereo
        # NO -ar flag: preserve native sample rate
        OUTPUT_WAV
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    if res.returncode != 0:
        print('ffmpeg FAILED:')
        print(res.stderr[-3000:])
        raise RuntimeError('Audio extraction failed')
    size_mb = os.path.getsize(OUTPUT_WAV) / 1e6
    print(f'Extracted in {elapsed:.1f}s -> {size_mb:.1f} MB')
    print(f'Output: {OUTPUT_WAV}')


In [ ]:
# ── Verify extracted file ─────────────────────────────────────────────────────
y, sr = librosa.load(OUTPUT_WAV, sr=None, mono=False)
print(f'Sample rate  : {sr} Hz')
print(f'Channels     : {y.shape[0] if y.ndim > 1 else 1}')
print(f'Duration     : {y.shape[-1]/sr:.2f}s')
print(f'Dtype        : {y.dtype}')
print(f'Peak amplitude: {float(y.max()):.4f}  (clipping if > 1.0)')

# Confirm SR matches probe
if sr != SOURCE_SAMPLE_RATE:
    print(f'WARNING: extracted SR {sr} differs from probed SR {SOURCE_SAMPLE_RATE}')
else:
    print(f'SR confirmed: {sr} Hz matches source probe')

SOURCE_SAMPLE_RATE = sr


In [ ]:
# ── Save metadata (read by all downstream notebooks) ─────────────────────────
meta = {
    'source_video':        SOURCE_VIDEO_PATH,
    'output_wav':          OUTPUT_WAV,
    'duration_seconds':    DURATION,
    'source_sample_rate':  SOURCE_SAMPLE_RATE,
    'source_language':     SOURCE_LANGUAGE,
    'language_display':    LANGUAGE_DISPLAY[SOURCE_LANGUAGE],
    'channels':            2,
}
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)
print('meta.json saved:')
print(json.dumps(meta, indent=2))


In [ ]:
import soundfile as sf
PREVIEW_SECS = 30
y_prev, sr_prev = sf.read(OUTPUT_WAV, frames=PREVIEW_SECS * SOURCE_SAMPLE_RATE)
print(f"Preview: first {PREVIEW_SECS}s of {DURATION/60:.1f} min")
display(Audio(y_prev.T, rate=sr_prev))